In [ ]:
!pip install -q pypdf sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 7.1 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y google-generativeai
!pip install -q -U google-genai

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 6.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.1 which is incompatible.


In [ ]:
import os
import re
import zipfile
import pandas as pd
import numpy as np

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from google import genai

In [ ]:
client = genai.Client(api_key="your_api_key")

In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello.")

print(response.text)

Hello! How can I help you today?


In [ ]:
zip_path = "/content/Reports.zip"
extract_folder = "/content/reports"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_folder)

print("Reports extracted successfully!")

Reports extracted successfully!


# Selecting 6 reports from jan-june 2026

In [ ]:
selected_months = [
    "january",
    "february",
    "march",
    "april",
    "may",
    "june"]

selected_reports = []

for root, folders, files_in_folder in os.walk(extract_folder):

    for filename in files_in_folder:

        filename_lower = filename.lower()

        if filename_lower.endswith(".pdf"):

            if ("2026" in filename_lower
                and "_en" in filename_lower
                and any(month in filename_lower for month in selected_months)):

                full_path = os.path.join(root, filename)

                selected_reports.append(full_path)

selected_reports = sorted(selected_reports)

print("Number of selected reports:", len(selected_reports))

for report in selected_reports:
    print(os.path.basename(report))

Number of selected reports: 6
april_monthly_report_2026_en.pdf
february_monthly_report_2026_en.pdf
january_monthly_report_2026_en.pdf
june_monthly_report_2026_en.pdf
march_monthly_report_2026_en.pdf
may_monthly_report_2026_en.pdf


# Month metadata retrival

In [ ]:
months = [
    "January", "February", "March",
    "April", "May", "June",
    "July", "August", "September",
    "October", "November", "December"]


def get_month(filename):
    filename = filename.lower()
    for month in months:
        if month.lower() in filename:
            return month
    return "Unknown"

# Remove repeated spaces/newlines

In [ ]:
def clean_text(text):

    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\bv\b", " ", text)

    return text.strip()

# chunking

In [ ]:
def chunk_text(text, max_words=200, overlap=40):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = start + max_words

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks

I chunked each page separately instead of combining the whole PDF first

Because I want the page metadata
If I combined the entire PDF first I would no longer know exactly which page a chunk came from

In [ ]:
all_chunks = []

for pdf_path in selected_reports:

    filename = os.path.basename(pdf_path)
    month = get_month(filename)
    reader = PdfReader(pdf_path)

    for page_index, page in enumerate(reader.pages):
        page_number = page_index + 1
        text = page.extract_text()

        if text is None:
            continue

        text = clean_text(text)

        if text == "":
            continue

        chunks = chunk_text(text)

        for chunk in chunks:

            all_chunks.append({

                "source": filename,
                "month": month,
                "year": 2026,
                "page": page_number,
                "text": chunk})


print("Number of reports:", len(selected_reports))
print("Total number of chunks:", len(all_chunks))

Number of reports: 6
Total number of chunks: 522


In [ ]:
chunks_df = pd.DataFrame(all_chunks)
chunks_df.head(10)

,source,month,year,page,text
0,april_monthly_report_2026_en.pdf,April,2026,1,Payment Systems Monthly Reports April 2026 Jor...
1,april_monthly_report_2026_en.pdf,April,2026,2,Table of Content 25 25 9 9 Systems Summary 3 3...
2,april_monthly_report_2026_en.pdf,April,2026,3,3 Systems Summary Monthly Report April 2026 04...
3,april_monthly_report_2026_en.pdf,April,2026,4,Summary vvv Number of Users 2.25M Number of Tr...
4,april_monthly_report_2026_en.pdf,April,2026,5,Summary vvv Number of Users 2.93M Number of Tr...
5,april_monthly_report_2026_en.pdf,April,2026,6,Summary Number of Users Number of Billers Numb...
6,april_monthly_report_2026_en.pdf,April,2026,7,Summary vvv Total Number of Transactions 1.07M...
7,april_monthly_report_2026_en.pdf,April,2026,8,Summary 8 Number of Cheques 438K Number of Ret...
8,april_monthly_report_2026_en.pdf,April,2026,9,9 Monthly Report April 2026 Instant Payment Sy...
9,april_monthly_report_2026_en.pdf,April,2026,10,Number of Transactions 72.62M Value of Transac...


# Text representation (embedding) :)

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts_for_embedding = []
for chunk in all_chunks:
    searchable_text = (
        f"{chunk['month']} "
        f"{chunk['year']} "
        f"{chunk['text']}")

    texts_for_embedding.append(searchable_text)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
chunk_emb = embed_model.encode(
    texts_for_embedding)

print("Embedding shape:", chunk_emb.shape)

Embedding shape: (522, 384)


# Cosine similarity + retrieval!!

In [ ]:
def retrieve_chunks(question, top_k=3):

    query_emb = embed_model.encode([question])
    similarities = cosine_similarity(query_emb,chunk_emb)[0]

    top_indices = similarities.argsort()[-top_k:][::-1]


    results = []
    for idx in top_indices:
        results.append({

            "score": similarities[idx],
            "source":
                all_chunks[idx]["source"],
            "month":
                all_chunks[idx]["month"],
            "year":
                all_chunks[idx]["year"],
            "page":
                all_chunks[idx]["page"],
            "text":
                all_chunks[idx]["text"]})
    return results

# Test retrieval b4 adding gemini!! This is important because RAG has two parts: retrieval and generation.
# I need to first prove that retrieval works

In [ ]:
question = (
    "How many eFAWATEERcom users"
    "were there in June 2026?")

results = retrieve_chunks(question,top_k=3)

for i, result in enumerate(results, start=1):

    print("result", i)

    print("Relevance score:",round(result["score"], 4))

    print("Report:",result["source"])

    print("Month:",result["month"])

    print("Page:",result["page"])

    print("Evidence:",result["text"])

result 1
Relevance score: 0.7645
Report: june_monthly_report_2026_en.pdf
Month: June
Page: 63
Evidence: eFAWATEERcom Users Number of eFAWATEERcom Users 99.2% 0.8 % Existing Users 5.25 M New Users 42.45K Total Users 5.29 M 63 06.2026
result 2
Relevance score: 0.7394
Report: june_monthly_report_2026_en.pdf
Month: June
Page: 62
Evidence: eFAWATEERcom Users Total Number of Users Over the Past 3 Months 5.21 M 5.25 M 5.29 M Apr-26 May-26 Jun-26 Increase in the number of users in the last month 0.8 % 62 06.2026
result 3
Relevance score: 0.7291
Report: may_monthly_report_2026_en.pdf
Month: May
Page: 63
Evidence: eFAWATEERcom Users Number of eFAWATEERcom Users 99.3 % 0.7 % Existing Users 5.21 M New Users 40.46K Total Users 5.25 M 63 05.2026


# RAG!!

In [ ]:
def rag_query(question, top_k=3):

    retrieved = retrieve_chunks(question,top_k)

    context = ""
    for result in retrieved:
        context += f"""

SOURCE REPORT: {result['source']}
MONTH: {result['month']} {result['year']}
PAGE: {result['page']}

TEXT:
{result['text']}

"""

    prompt = f"""
Use ONLY the information in the retrieved
JoPACC report context to answer the question.

Do not use external knowledge.

Do not invent information.

If the answer is not available in the context,
say: "Not found in the retrieved documents."

Give a clear and concise answer.

At the end of the answer, mention the
report month and page used.

CONTEXT:
{context}

QUESTION:{question}

ANSWER:
    """

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, retrieved

# Function to show answer AND evidence

In [ ]:

def ask_question(question):

    answer, retrieved = rag_query(question,top_k=3)

    print("=" * 70)
    print("question:")

    print(question)

    print("\nanswer:")

    print(answer)

    print("\nretrieved evidence:")


    for i, result in enumerate(retrieved,start=1):

        print("Result:", i)

        print("Relevance score:",round(result["score"], 4))

        print("Report:",result["source"])

        print("Month:",result["month"])

        print("Page:", result["page"])

        print("Evidence:",result["text"])



    return answer, retrieved

# testing :)

In [ ]:
answer1, docs1 = ask_question("How many eFAWATEERcom users were there in June 2026?")

question:
How many eFAWATEERcom users were there in June 2026?

answer:
There were 5.29 million total eFAWATEERcom users in June 2026.

Report Month: June 2026
Page: 63

retrieved evidence:
Result: 1
Relevance score: 0.8164
Report: june_monthly_report_2026_en.pdf
Month: June
Page: 63
Evidence: eFAWATEERcom Users Number of eFAWATEERcom Users 99.2% 0.8 % Existing Users 5.25 M New Users 42.45K Total Users 5.29 M 63 06.2026
Result: 2
Relevance score: 0.8146
Report: june_monthly_report_2026_en.pdf
Month: June
Page: 62
Evidence: eFAWATEERcom Users Total Number of Users Over the Past 3 Months 5.21 M 5.25 M 5.29 M Apr-26 May-26 Jun-26 Increase in the number of users in the last month 0.8 % 62 06.2026
Result: 3
Relevance score: 0.8035
Report: february_monthly_report_2026_en.pdf
Month: February
Page: 62
Evidence: eFAWATEERcom Users Total Number of Users Over the Past 3 Months 5.04 M 5.09 M 5.13 M Dec-25 Jan-26 Feb-26 Increase in the number of users in the last month 0.9 % 62 02.2026


In [ ]:
answer2, docs2 = ask_question(
"Did the value of eFAWATEERcom transactions "
    "increase or decrease in May 2026?")

question:
Did the value of eFAWATEERcom transactions increase or decrease in May 2026?

answer:
The value of eFAWATEERcom transactions decreased in May 2026 (by 16.5%).

Report Month: May 2026
Page: 68

retrieved evidence:
Result: 1
Relevance score: 0.8254
Report: june_monthly_report_2026_en.pdf
Month: June
Page: 68
Evidence: eFAWATEERcom Value of Transactions Total Value of Transactions Over the Past 3 Months 1.43 B 1.19 B 1.26 B Apr-26 May-26 Jun-26 68 JOD JOD JOD 06.2026 Increase in the value of transactions in the last month 6%
Result: 2
Relevance score: 0.8243
Report: march_monthly_report_2026_en.pdf
Month: March
Page: 68
Evidence: eFAWATEERcom Value of Transactions Total Value of Transactions Over the Past 3 Months 1.39 B 1.24 B 1.26 B Jan-26 Feb-26 Mar-26 68 JOD JOD JOD 03.2026 Increase in the value of transactions in the last month 1.5 %
Result: 3
Relevance score: 0.8165
Report: may_monthly_report_2026_en.pdf
Month: May
Page: 68
Evidence: eFAWATEERcom Value of Transactions Tota

In [ ]:
answer3, docs3 = ask_question(
    "What was the number of ACH Jordanian Dinar "
    "transactions in March 2026?")

question:
What was the number of ACH Jordanian Dinar transactions in March 2026?

answer:
The number of ACH Jordanian Dinar transactions in March 2026 was 1.12 M.

Report Month: March 2026
Page: 74

retrieved evidence:
Result: 1
Relevance score: 0.8872
Report: march_monthly_report_2026_en.pdf
Month: March
Page: 74
Evidence: ACH Transactions Total Number of Jordanian Dinar Transactions Over the Past 3 Months 1.03 M 1.08 M 1.12 M Jan-26 Feb-26 Mar-26 74 Increase in the number of transactions in the last month 4.3 % 03.2026
Result: 2
Relevance score: 0.8859
Report: february_monthly_report_2026_en.pdf
Month: February
Page: 74
Evidence: ACH Transactions Total Number of Jordanian Dinar Transactions Over the Past 3 Months 1.19 M 1.03 M 1.08 M Dec-25 Jan-26 Feb-26 74 02.2026 Increase in the number of transactions in the last month 4.3 %
Result: 3
Relevance score: 0.8785
Report: january_monthly_report_2026_en.pdf
Month: January
Page: 74
Evidence: ACH Transactions Total Number of Jordanian Dinar